# DeepFashion Database: Preprocessing
#### DeepFashion 'Anno_fine' files

Goals:
- Explore the provided train, validation and test splits to confirm sizes
- Merge the training files into one dataframe
- Check indices are aligned

In [1]:
import pandas as pd

In [2]:
# Read the train, val and test splits
fine_train_names_df = pd.read_csv(
    "/Users/bcjcal/Repos/207/Final/Data/DeepFashion/Anno_fine/train.txt",
    sep=r"\s+",
    names=["img_name"]
)

fine_val_names_df = pd.read_csv(
    "/Users/bcjcal/Repos/207/Final/Data/DeepFashion/Anno_fine/val.txt",
    sep=r"\s+",
    names=["img_name"]
)

fine_test_names_df = pd.read_csv(
    "/Users/bcjcal/Repos/207/Final/Data/DeepFashion/Anno_fine/test.txt",
    sep=r"\s+",
    names=["img_name"]
)

# Confirm the given train, val, and test split sizes
print(f"Train size: {len(fine_train_names_df)}")
print(f"Validation size: {len(fine_val_names_df)}")
print(f"Test size: {len(fine_test_names_df)}")

Train size: 14000
Validation size: 2000
Test size: 4000


In [3]:
# Read training category labels
fine_train_category_df = pd.read_csv(
    "/Users/bcjcal/Repos/207/Final/Data/DeepFashion/Anno_fine/train_cate.txt",
    sep=r"\s+",
    names=["category_raw"]
)

# Read attribute lookup for the attribute columns
fine_attr_lookup_df = pd.read_csv(
    "/Users/bcjcal/Repos/207/Final/Data/DeepFashion/Anno_fine/list_attr_cloth.txt",
    sep=r"\s+",
    skiprows=1,
    header=0
)

# Read training attribute labels
fine_train_attr_df = pd.read_csv(
    "/Users/bcjcal/Repos/207/Final/Data/DeepFashion/Anno_fine/train_attr.txt",
    sep=r"\s+",
    names=list(fine_attr_lookup_df["attribute_name"])
)

# Read training bounding boxes
fine_train_bbox_df = pd.read_csv(
    "/Users/bcjcal/Repos/207/Final/Data/DeepFashion/Anno_fine/train_bbox.txt",
    sep=r"\s+",
    names=["bbox_x1", "bbox_y1", "bbox_x2", "bbox_y2"]
)

In [4]:
# Confirm training images, categories, attributes, and bounding boxes shapes before merging
print(f"Training images shape {fine_train_names_df.shape}")
print(f"Training categories shape {fine_train_category_df.shape}")
print(f"Training attributes shape {fine_train_attr_df.shape}")
print(f"Training bounding boxes shape {fine_train_bbox_df.shape}")

Training images shape (14000, 1)
Training categories shape (14000, 1)
Training attributes shape (14000, 26)
Training bounding boxes shape (14000, 4)


In [5]:
# Combine training images, categories, attributes, and bounding boxes into singular training dataframe
train_df = pd.concat([fine_train_names_df, fine_train_category_df, fine_train_bbox_df, fine_train_attr_df], axis=1)

# Print shape and columns and preview train_df
print(f"Merged training dataframe shape {train_df.shape}")
print(f"Merged training dataframe columns {train_df.columns}")
train_df.head()

Merged training dataframe shape (14000, 32)
Merged training dataframe columns Index(['img_name', 'category_raw', 'bbox_x1', 'bbox_y1', 'bbox_x2', 'bbox_y2',
       'floral', 'graphic', 'striped', 'embroidered', 'pleated', 'solid',
       'lattice', 'long_sleeve', 'short_sleeve', 'sleeveless', 'maxi_length',
       'mini_length', 'no_dress', 'crew_neckline', 'v_neckline',
       'square_neckline', 'no_neckline', 'denim', 'chiffon', 'cotton',
       'leather', 'faux', 'knit', 'tight', 'loose', 'conventional'],
      dtype='object')


,img_name,category_raw,bbox_x1,bbox_y1,bbox_x2,bbox_y2,floral,graphic,striped,embroidered,...,no_neckline,denim,chiffon,cotton,leather,faux,knit,tight,loose,conventional
0,img/Sweet_Crochet_Blouse/img_00000070.jpg,3,66,75,241,293,0,0,0,1,...,1,0,1,0,0,0,0,0,0,1
1,img/Classic_Pencil_Skirt/img_00000010.jpg,33,65,88,132,218,0,0,0,0,...,1,0,0,1,0,0,0,1,0,0
2,img/Strapless_Diamond_Print_Dress/img_00000038...,41,75,43,176,300,0,1,0,0,...,1,0,0,1,0,0,0,0,0,1
3,img/Mid-Rise_-_Acid_Wash_Skinny_Jeans/img_0000...,26,64,1,129,273,0,0,0,0,...,1,1,0,0,0,0,0,1,0,0
4,img/Zippered_Single-Button_Blazer/img_00000078...,2,1,12,257,300,0,0,0,0,...,0,0,0,1,0,0,0,0,0,1


In [6]:
# Check current range for categories
print(f"Raw category range ({train_df['category_raw'].min()}, {train_df['category_raw'].max()})")

# Update so indexing begins at 0, not 1
train_df["category_id"] = train_df["category_raw"] - 1

# Check work
print(f"Cleaned category range ({train_df['category_id'].min()}, {train_df['category_id'].max()})")


Raw category range (1, 48)
Cleaned category range (0, 47)


In [7]:
# Add string category names to training data
fine_category_lookup_df = pd.read_csv(
    "/Users/bcjcal/Repos/207/Final/Data/DeepFashion/Anno_fine/list_category_cloth.txt",
    sep=r"\s+",
    skiprows=1,
    header=0
)

# Adjust for train_cate.txt index at 1
fine_category_lookup_df["category_raw"] = fine_category_lookup_df.index + 1

# Join
train_df = train_df.merge(
    fine_category_lookup_df,
    on="category_raw",
    how="left"
)

print(f"The training shape is now {train_df.shape}")
print(f"The training dataframe columns are now {train_df.columns}")


The training shape is now (14000, 35)
The training dataframe columns are now Index(['img_name', 'category_raw', 'bbox_x1', 'bbox_y1', 'bbox_x2', 'bbox_y2',
       'floral', 'graphic', 'striped', 'embroidered', 'pleated', 'solid',
       'lattice', 'long_sleeve', 'short_sleeve', 'sleeveless', 'maxi_length',
       'mini_length', 'no_dress', 'crew_neckline', 'v_neckline',
       'square_neckline', 'no_neckline', 'denim', 'chiffon', 'cotton',
       'leather', 'faux', 'knit', 'tight', 'loose', 'conventional',
       'category_id', 'category_name', 'category_type'],
      dtype='object')


In [ ]:
# train_df.to_csv(
#     "/Users/bcjcal/Repos/207/Final/Data/DeepFashion/clean_train.csv",
#     index=False
# )